# 🧬 Aircheck Workshop: Using Machine Learning to Find Hits

**AIRCHECK Workshop 2026** · Training, evaluating and screening small molecules with models
built on chemical fingerprints.

This notebook walks the whole path: load the DEL screening data, train a LightGBM model on
molecular fingerprints, evaluate it honestly, screen a compound library, then filter and
cluster the nominees down to a shortlist worth taking to the bench.

---

# 📦 Section 1 · Install and Import Dependencies

In this section we install the packages the workshop needs for chemical data processing
and machine learning.

In [ ]:
# This notebook runs both in Google Colab and from a local clone of the repository.
# In Colab it clones the repository so the workshop data is available; locally it
# simply locates the repository root. Either way the paths below are the same.
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ShagReza/Aircheck-Workshop-2026.git"
REPO_NAME = "Aircheck-Workshop-2026"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO_ROOT = Path(REPO_NAME).resolve()
else:
    REPO_ROOT = Path.cwd().resolve()
    while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Running in Colab" if IN_COLAB else "Running locally")
print(f"Repository root: {REPO_ROOT}")
print(f"Data files:      {sorted(p.name for p in DATA_DIR.glob('*.parquet'))}")

In [ ]:
# Install every package the workshop needs, as listed in the repository's requirements.txt.
# In Colab this installs into the runtime. Locally we assume you already created a virtual
# environment with `python -m pip install -r requirements.txt`, so nothing is installed here.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_ROOT / "requirements.txt")], check=True)
    print("Requirements installed.")
else:
    print("Local run - install requirements with:")
    print(f"    python -m pip install -r {REPO_ROOT / 'requirements.txt'}")

In [ ]:
# Import libraries:
import pandas as pd
import numpy as np
import joblib
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem
import os

---

# 📂 Section 2 · Load the Workshop Data

The datasets ship with this repository, so there is nothing to download and no Google Drive
to mount. The bootstrap cell above already cloned the repository (in Colab) or found it on
disk (locally), so the files are ready to read from `data/`.

| File | Compounds | What it is | Used in this run |
|---|---|---|---|
| `sample-train.parquet` | 4,000 | DEL screen against WDR91, balanced 50/50 on `LABEL` | **training set** |
| `sample-test-2.parquet` | 5,000 | labelled, carries `SMILES`, only **9** actives | **evaluation and screening set** |
| `sample-test.parquet` | 2,000 | held-out slice of the DEL screen, also balanced | not used |
| `sample-screen.parquet` | 5,000 | carries `SMILES`, no labels | not used |

> **Why `sample-test-2` does double duty**
>
> It is the only file that has **both** `SMILES` and `LABEL`. Using it as the screening set
> means we can do something a real screen cannot: check afterwards whether the compounds the
> model nominated are the ones that were actually active. Swap `df_screen` to
> `sample-screen.parquet` when you want the genuine unlabelled experience.

`df_train` is already balanced, so unlike previous years there is no downsampling step here.

Note how different the training and evaluation sets look. We train on a balanced set, but the
evaluation set has **9 actives in 5,000 compounds (0.18%)** — which is what screening actually
looks like. Keep that in mind when you read the metrics: a model that predicts "inactive" for
everything scores 99.8% accuracy and is completely useless.

In [ ]:
# Read the datasets into Pandas DataFrames.
# sample-train is the training set; sample-test-2 serves as BOTH the evaluation set
# and the screening set, because it is the only file carrying SMILES and labels.
df_train = pd.read_parquet(DATA_DIR / "sample-train.parquet")     # DEL screen, balanced, labelled
df_test2 = pd.read_parquet(DATA_DIR / "sample-test-2.parquet")    # labelled, with SMILES, 9 actives

df_test = df_test2      # the same frame, used in the "evaluation" role
df_screen = df_test2    # ...and in the "screening" role

for name, frame in [("df_train", df_train), ("df_test2 / df_test / df_screen", df_test2)]:
    actives = int(frame["LABEL"].sum()) if "LABEL" in frame.columns else None
    print(f"{name:<32} {frame.shape[0]:>5} rows x {frame.shape[1]:>2} columns"
          + (f"   actives: {actives}" if actives is not None else "   (unlabelled)"))


## Training set

In [ ]:
# Display first few rows of the train dataset
df_train.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_train.columns.tolist()
print(column_names_list)

## Screening set

In [ ]:
# Display first few rows of the screening dataset
df_screen.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_screen.columns.tolist()
print(column_names_list)

---

# 🧬 Section 3 · Selecting Fingerprint Columns and ML Labels

In [ ]:
# Function to turn a fingerprint column into a feature matrix
def process_data(X, column_name):
    """Stack a fingerprint column into a 2D array of shape (n_molecules, n_bits).

    The fingerprints are stored as arrays of counts, so this is just a stack - there is no
    string to parse. We cast to float32 because that is what LightGBM expects.
    """
    return np.stack(X[column_name].to_numpy()).astype(np.float32)


# List of available fingerprint column names
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
selected_fps = 'ECFP4'  # Replace with desired fingerprints

# Build the feature matrices for each dataset
TrainData = process_data(df_train, selected_fps)
TestData = process_data(df_test, selected_fps)
Test2Data = process_data(df_test2, selected_fps)
ScreenData = process_data(df_screen, selected_fps)

# Labels come from the 'LABEL' column
TrainLabel = df_train['LABEL']
TestLabel = df_test['LABEL']
Test2Label = df_test2['LABEL']

print(f"TrainData:  {TrainData.shape}")
print(f"TestData:   {TestData.shape}")
print(f"Test2Data:  {Test2Data.shape}")
print(f"ScreenData: {ScreenData.shape}")

---

# ⚙️ Section 4 · Define the ML Model

## LightGBM

Light Gradient Boosting Machine is a fast, scalable gradient boosting framework. It builds
decision trees iteratively, each one correcting the errors of the ones before it. Unlike
older gradient boosting implementations it uses a histogram-based approach, which speeds up
training considerably on large datasets, and it handles categorical features directly.

> **Why it suits this problem.** Fingerprints are wide, sparse, tabular count vectors, and
> gradient boosting is consistently the strongest family on tabular data. This is the same
> reasoning laid out in Section 2 of the ML introduction notebook.

In [ ]:
from lightgbm import LGBMClassifier

# Initialize model with detailed hyperparameters using default values
model = LGBMClassifier(
    n_estimators=100,  # Number of boosting iterations (trees)
    n_jobs=1,  # Number of parallel jobs (1 for no parallelism)
    learning_rate=0.1,  # Learning rate
    max_depth=-1,  # No limit on maximum depth of trees
    min_child_samples=20,  # Minimum samples at leaf node
    reg_lambda=0.0,  # L2 regularization (no regularization)
    reg_alpha=0.0,  # L1 regularization (no regularization)
    num_leaves=31,  # Number of leaves in each tree
    max_bin=255,  # Maximum number of bins
    subsample=1.0,  # Subsample ratio for training data (use all data)
    colsample_bytree=1.0,  # Subsample ratio for features (use all features)
    random_state=42,  # Random seed for reproducibility
    boosting_type='gbdt',  # Boosting type (Gradient Boosting Decision Tree)
    min_split_gain=0.0,  # Minimum loss reduction required to make a further partition
    verbose=-1,  # Quieten LightGBM's per-tree logging
)

# Model is now initialized with default hyperparameters

---

# 🔁 Section 5 · Training the Model and Evaluating on Cross-Validation

## Cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)

# Function to train the model and compute all classification metrics
def train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test):
    """Train a LightGBM model and compute accuracy, precision, recall, F1, AUC, MCC, and Kappa."""
    model = LGBMClassifier(random_state=42, verbose=-1)
    model.fit(CrossVal_data_train, CrossVal_label_train)

    y_pred = model.predict(CrossVal_data_test)
    y_scores = model.predict_proba(CrossVal_data_test)[:, 1]  # Probability for positive class

    metrics = {
        "Accuracy": accuracy_score(CrossVal_label_test, y_pred),
        "Precision": precision_score(CrossVal_label_test, y_pred, zero_division=0),
        "Recall": recall_score(CrossVal_label_test, y_pred),
        "F1-Score": f1_score(CrossVal_label_test, y_pred),
        "AUC-ROC": roc_auc_score(CrossVal_label_test, y_scores) if len(set(CrossVal_label_test)) > 1 else None,
        "MCC": matthews_corrcoef(CrossVal_label_test, y_pred),
        "Cohen's Kappa": cohen_kappa_score(CrossVal_label_test, y_pred),
    }

    return model, metrics

# Cross-validation
Nfold = 5
skf = StratifiedKFold(n_splits=Nfold, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(TrainData, TrainLabel)):
    # Renaming the partitions inside the loop
    CrossVal_data_train, CrossVal_data_test = TrainData[train_idx], TrainData[test_idx]
    CrossVal_label_train, CrossVal_label_test = TrainLabel.iloc[train_idx], TrainLabel.iloc[test_idx]

    # Now using the renamed variables for both train and test data
    _, metrics = train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test)
    fold_metrics.append(metrics)

    # Print fold metrics
    print(f"Fold {fold_idx+1} Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print("-" * 100)

In [ ]:
# Compute average metrics across folds
avg_metrics = {metric: np.mean([fold[metric] for fold in fold_metrics]) for metric in fold_metrics[0]}

# Print average metrics
print("\nAverage Metrics across all folds:")
for metric, value in avg_metrics.items():
    print(f"{metric}: {value:.4f}")

## Train the final model

In [ ]:
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score


# Function to train the final model on the entire dataset
def train_final_model(X, y):
    final_model = LGBMClassifier(random_state=42, verbose=-1)
    final_model.fit(X, y)
    model_filename = RESULTS_DIR / "final_model.pkl"
    joblib.dump(final_model, model_filename)
    print(f"Final model saved as {model_filename}.")
    return final_model


# Train final model on entire dataset
final_model = train_final_model(TrainData, TrainLabel)


def report(name, X, y):
    pred = final_model.predict(X)
    scores = final_model.predict_proba(X)[:, 1]
    print(f"\n{name}  ({int(y.sum())} actives out of {len(y)})")
    print(f"  Accuracy:          {accuracy_score(y, pred):.4f}")
    print(f"  AUC-ROC:           {roc_auc_score(y, scores):.4f}")
    print(f"  Average precision: {average_precision_score(y, scores):.4f}")
    print(f"  Predicting 'inactive' for everything would score "
          f"{accuracy_score(y, np.zeros_like(y)):.4f} accuracy")


# df_test and df_test2 are the same frame in this configuration, so report it once.
# This is the realistic picture: 9 actives among 5,000 compounds.
report("Realistic evaluation set (sample-test-2)", Test2Data, Test2Label)

---

# 🔬 Virtual Screening

Sections 1 to 5 built a model and checked that it works. From here we put it to use: score
a library of compounds, combine several fingerprints for robustness, then filter and cluster
what comes out into a shortlist.

---

# 🎯 Section 6 · Screen Test Compounds

We can now score new compounds to predict their activity, by passing their fingerprints
through the trained model.

In [ ]:
def evaluate_model(model, X_test):
    """Evaluate the model on the screening set and return prediction scores."""
    y_scores = model.predict_proba(X_test)[:, 1]  # Probability for positive class
    return np.round(y_scores, 3)


predictions = evaluate_model(final_model, ScreenData)

# Create a DataFrame with SMILES and prediction scores
prediction_df = pd.DataFrame({
    'SMILES': df_screen["SMILES"],
    'Prediction_Score': predictions
})

# Sort the DataFrame by prediction score in descending order
prediction_df_sorted = prediction_df.sort_values(by='Prediction_Score', ascending=False)

# Keep only those with score > 0.5 as Possible Nominees
nominees = prediction_df_sorted[prediction_df_sorted['Prediction_Score'] > 0.5]

# Get the number of nominees
num_nominees = nominees.shape[0]
print(f"\nNumber of Possible Nominees: {num_nominees}")

# Print the top 10 highest-ranked predictions
print("Top 10 Predictions:")
print(prediction_df_sorted.head(10))

---

# 🧩 Section 7 · Using an Ensemble of Models

This improves reliability by training one model per fingerprint (ECFP4, MACCS, RDK) instead
of relying on a single representation. For each compound we take the mean prediction across
models, their standard deviation, and a confidence score.

The final score is **mean minus standard deviation**, so compounds the models disagree about
are pushed down the list. That is deliberately conservative — see the threshold comparison
below for what it costs you.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMClassifier

# List of fingerprint columns to train on
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
fingerprint_columns = ['ECFP4', 'MACCS', 'RDK']

# Dictionary to store trained models
trained_models = {}

# Train models for each fingerprint column
for fp in fingerprint_columns:
    print(f"Training model for {fp}...")
    TrainData_fp = process_data(df_train, fp)

    model = LGBMClassifier(random_state=42, verbose=-1)
    model.fit(TrainData_fp, TrainLabel)

    model_filename = RESULTS_DIR / f"final_model_{fp}.pkl"
    joblib.dump(model, model_filename)
    print(f"Model for {fp} saved as {model_filename}.")

    trained_models[fp] = model


# Function to evaluate models
def evaluate_model(model, X_test):
    return model.predict_proba(X_test)[:, 1]  # Probability for positive class


# Store predictions for each model
all_predictions = {}

for fp in fingerprint_columns:
    print(f"Evaluating model for {fp}...")
    ScreenData_fp = process_data(df_screen, fp)
    all_predictions[fp] = evaluate_model(trained_models[fp], ScreenData_fp)

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(all_predictions, index=df_screen.index)
predictions_df['Mean_Prediction'] = predictions_df[fingerprint_columns].mean(axis=1)
predictions_df['Std_Dev'] = predictions_df[fingerprint_columns].std(axis=1)
predictions_df['Confidence_Score'] = 1 - predictions_df['Std_Dev']  # Higher means more confident
predictions_df['Final_Score'] = predictions_df['Mean_Prediction'] - predictions_df['Std_Dev']

# Add SMILES column
predictions_df['SMILES'] = df_screen['SMILES']

# Sort by Final Score
predictions_df_sorted = predictions_df.sort_values(by='Final_Score', ascending=False)

# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.5]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))

In [ ]:
# How many nominees does each threshold give, and how many real actives survive?
# We can only ask this because sample-test-2 is labelled - a true screening set is not.
known_actives = int(df_screen["LABEL"].sum())

print(f"{'threshold':>10}  {'nominees':>9}  actives kept (of {known_actives})")
for thr in [0.5, 0.4, 0.3, 0.2]:
    selected = predictions_df_sorted["Final_Score"] > thr
    kept = int(df_screen.loc[predictions_df_sorted.index[selected], "LABEL"].sum())
    print(f"{thr:>10.1f}  {int(selected.sum()):>9d}  {kept}")

# Final_Score is mean-minus-std, so it punishes disagreement between fingerprints.
# A strict cut-off is not free: it discards real hits the models argued about.
THRESHOLD = 0.3
nominees = predictions_df_sorted[predictions_df_sorted["Final_Score"] > THRESHOLD]

print(f"\nCarrying {nominees.shape[0]} nominees forward at Final_Score > {THRESHOLD}.")
print("Top 10:")
print(predictions_df_sorted.head(10))

---

# 🧪 Section 8 · Apply Medicinal Chemistry Filters

This part applies drug-likeness filters to the nominees, using three standard rules. Molecules
passing all three are kept as final candidates.

- **Lipinski’s Rule of 5** — molecular weight, lipophilicity (logP), hydrogen bond
  donors and acceptors, and rotatable bonds.
- **Ghose filter** — molecular weight, logP, atom count and molar refractivity, for
  favourable pharmacokinetics.
- **Veber rule** — limited rotatable bonds and acceptable topological polar surface area,
  for oral bioavailability.

> **Note:** this is a simplified version of a much larger family of drug-likeness filters.

In [ ]:
from rdkit import Chem
import rdkit.Chem.Descriptors as Descriptors

class SimplifiedDrugFilters:
    def __init__(self):
        pass

    @staticmethod
    def fetch_attributes(molecule):
        return {
            "molecular_weight": Descriptors.ExactMolWt(molecule),
            "logp": Descriptors.MolLogP(molecule),
            "h_bond_donor": Descriptors.NumHDonors(molecule),
            "h_bond_acceptors": Descriptors.NumHAcceptors(molecule),
            "rotatable_bonds": Descriptors.NumRotatableBonds(molecule),
            "num_atoms": Chem.rdchem.Mol.GetNumAtoms(molecule),
            "molar_refractivity": Chem.Crippen.MolMR(molecule),
            "topo_surface_area": Chem.QED.properties(molecule).PSA
        }

    def filter(self, smiles):
        results = {"lipinski": [], "ghose": [], "veber": [], "pass_all_filters": []}
        molecules = [Chem.MolFromSmiles(i) for i in smiles]

        for i, mol in enumerate(molecules):
            props = self.fetch_attributes(mol)

            # Lipinski Rule of 5
            lipinski = (props["molecular_weight"] <= 500 and props["logp"] <= 5 and
                        props["h_bond_donor"] <= 5 and props["h_bond_acceptors"] <= 10 and
                        props["rotatable_bonds"] <= 5)

            # Ghose Filter
            ghose = (160 <= props["molecular_weight"] <= 480 and -0.4 <= props["logp"] <= 5.6 and
                     20 <= props["num_atoms"] <= 70 and 40 <= props["molar_refractivity"] <= 130)

            # Veber Rule
            veber = (props["rotatable_bonds"] <= 10 and props["topo_surface_area"] <= 140)

            results["lipinski"].append(lipinski)
            results["ghose"].append(ghose)
            results["veber"].append(veber)
            results["pass_all_filters"].append(all([lipinski, ghose, veber]))

        return results

In [ ]:
# Apply drug design filters to the selected nominees
filter = SimplifiedDrugFilters()
filter_results = pd.DataFrame(filter.filter(nominees["SMILES"].tolist()), index=nominees.index)

# Merge the filter results with nominees
nominees_filtered = pd.merge(nominees, filter_results, left_index=True, right_index=True)
print("Top 10 Predictions:")
print(nominees_filtered.head(10))

nominees_filtered = nominees_filtered[nominees_filtered["pass_all_filters"] == True]
num_nominees_filtered = nominees_filtered.shape[0]
print(f"\nNumber of Filtered Nominees: {num_nominees_filtered}")

---

# 🗂️ Section 9 · Cluster and Select Compounds for Laboratory Testing

Finally, we use similarity-based clustering to pick a *diverse* shortlist from the top hits.
Nominating twenty near-identical molecules wastes twenty assays; nominating twenty different
scaffolds tests twenty ideas.

## Step 1 · Generate molecular fingerprints

Convert each molecule, given as a SMILES string, into a **Morgan fingerprint** with RDKit's
`AllChem.GetMorganFingerprintAsBitVect`. That produces a binary vector encoding the structure.

## Step 2 · Cluster with LeaderPicker

Apply the **LeaderPicker** algorithm to those fingerprints to find "leader" molecules, which
act as cluster centroids. The `thresh` parameter sets how similar a molecule must be to a
leader to join its cluster.

## Step 3 · Assign molecules to clusters

Compute the **Tanimoto similarity** between every molecule and each leader, then assign each
molecule to the cluster whose leader it most resembles. `assignPointsToClusters` does this.

## Step 4 · Select representatives and sort

Within each cluster, take a subset (about 1/20th of the cluster). Selection favours molecules
unlike those already picked, which keeps the shortlist diverse. Results are then sorted by
prediction score and cluster id.

In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from rdkit import Chem

def assignPointsToClusters(picks, fps):
    clusters = defaultdict(list)
    for i, idx in enumerate(picks):
        clusters[i].append(idx)
    sims = np.zeros((len(picks), len(fps)))
    for i in tqdm(range(len(picks))):
        pick = picks[i]
        sims[i, :] = DataStructs.BulkTanimotoSimilarity(fps[pick], fps)
        sims[i, i] = 0  # Don't compare the molecule with itself
    best = np.argmax(sims, axis=0)
    for i, idx in enumerate(best):
        if i not in picks:
            clusters[idx].append(i)
    return clusters

# nominees_filtered holds the compounds that survived the medicinal chemistry filters,
# together with their SMILES and Final_Score.

# Generate Morgan fingerprints using AllChem
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(smi)) for smi in tqdm(nominees_filtered["SMILES"])]

In [ ]:
# Perform clustering using LeaderPicker
lp = rdSimDivPickers.LeaderPicker()
thresh = 0.65  # Minimum distance between cluster centroids
picks = lp.LazyBitVectorPick(fps, len(fps), thresh)  # Using the ExplicitBitVect fingerprints from AllChem
clusters = assignPointsToClusters(picks, fps)

# Assign cluster ids to the prediction_df based on the indices from the clusters
cluster_ids = np.zeros(len(nominees_filtered))  # Initialize cluster_ids for the entire prediction_df

# Make sure to correctly assign the cluster ids
for key, val in clusters.items():
    cluster_ids[val] = key  # Assign the cluster ID to the correct indices

# Add the cluster ids to the prediction_df
nominees_filtered['cluster_id'] = cluster_ids

# Sort the results by Prediction Score and Cluster ID
#nominees_filtered.sort_values(by=["Final_Score", "cluster_id"], ascending=[False, True], inplace=True)

num_clusters = len(set(nominees_filtered["cluster_id"]))
print(f"Number of clusters generated: {num_clusters}")

# Sort the dataframe by Final_Score in descending order
nominees_filtered.sort_values(by=["Final_Score"], ascending=False, inplace=True)

# Select one nominee per cluster: the one with the highest score
best_nominees = nominees_filtered.groupby("cluster_id").first().reset_index()

# Print the selected nominees (one per cluster)
print(best_nominees.head(10))

I will edit this:

## **How to submit results**
save prediction in CSV format in our GCP bucket
With there TeamName like Team1.csv

Other ideas that I may add:

- Confusion matrix and other visualizations for this part
- Introducing other ML models they can use
- MLflow?
- Dummy test to show importance of mertics
- Add metrics to the ensemble one.

!!!! For the complains in the Bootcamp we had (simpler models, better results) --> We may want to give people some development data